In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [2]:
def prepare_raw_isot(isot_true_path: str, isot_fake_path: str):
    isot_true_df = pd.read_csv(isot_true_path)
    isot_true_df['label'] = 1
    isot_fake_df = pd.read_csv(isot_fake_path)
    isot_fake_df['label'] = 0

    df_isot_combined = pd.concat([isot_true_df, isot_fake_df], ignore_index=True)
    df_isot_combined = df_isot_combined[['text', 'label']]
    # Shuffling 5 times using a loop with varying random seeds
    for i in range(5):
        # Generates a new seed for each pass (e.g., 42, 43, 44, 45, 46)
        current_seed = 42 + i
        df_isot_combined = df_isot_combined.sample(frac=1, random_state=current_seed).reset_index(drop=True)
    df_isot_combined.dropna(inplace=True)
    df_isot_combined.drop_duplicates(inplace=True)
    df_isot_combined = df_isot_combined.reset_index(drop=True)
    return df_isot_combined

In [3]:

def load_and_merge_datasets(wakfake_pth:str, isot_dataset: pd.DataFrame = None):
    logger.info(f'Loading WAKFake dataset...')
    wf_df = pd.read_csv(wakfake_pth)
    wf_df = wf_df[['text', 'label']].dropna().reset_index(drop=True)

    wf_df['label'] = wf_df['label'].map({0: 1, 1: 0})

    if isot_dataset is not None:
        logger.info(f'Combining WAKFake and ISOT datasets into one dataset...')
        df_combined = pd.concat([wf_df, isot_dataset], ignore_index=True)
        df_combined.drop_duplicates(inplace=True)
        df_combined = df_combined.reset_index(drop=True)
        return df_combined

    return wf_df

In [4]:
isot_true = '../data/raw/ISOT_True.csv'
isot_fake = '../data/raw/ISOT_Fake.csv'
isot_df = prepare_raw_isot(isot_fake_path=isot_fake, isot_true_path=isot_true)

df = load_and_merge_datasets('../data/raw/WELFake.csv', isot_df)
print(df.shape)
df.head()

INFO:__main__:Loading WAKFake dataset...
INFO:__main__:Combining WAKFake and ISOT datasets into one dataset...



label=1 sample (should now be REAL news):
A dozen politically active pastors came here for a private dinner Friday night to hear a conversion story unique in the context of presidential politics: how Louisiana Gov. Bobby Jindal traveled from Hinduism to Protestant Christianity and, ultimately, became what he calls an “evangelical Catholic.”

label=0 sample (should now be FAKE news):
No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about t
(62719, 2)


,text,label
0,No comment is expected from Barack Obama Membe...,0
1,Did they post their votes for Hillary already?,0
2,"Now, most of the demonstrators gathered last ...",0
3,A dozen politically active pastors came here f...,1
4,"The RS-28 Sarmat missile, dubbed Satan 2, will...",0


In [17]:
def create_stratified_splits(
        df: pd.DataFrame,
        target_col: str = 'label',
        test_size: float = 0.15,
        val_size: float = 0.15
):
    # Creating a strict 70:15:15 stratified train/val/test split to preserve class distribution.
    logger.info("Executing stratified train/test/val splits...")

    combined_test_size = test_size + val_size
    train_df, temp_df = train_test_split(
        df,
        test_size=combined_test_size,
        stratify=df[target_col],
        random_state=42
    )

    relative_val_size = val_size / combined_test_size
    val_df, test_df = train_test_split(
        temp_df,
        test_size=1.0 - relative_val_size,
        stratify=temp_df[target_col],
        random_state=42
    )
    logger.info(f"Split completed. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    return train_df, val_df, test_df